In [1]:
#packages to include
#pip install torch==2.10.0 torchvision==0.25.0 torchaudio==2.10.0 --index-url https://download.pytorch.org/whl/cu126

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
import torch.nn.functional as F

In [3]:
class PrunableLinear(nn.Module):
    def __init__(self, in_features, out_features):
        super(PrunableLinear, self).__init__()

        # Standard weight parameter
        self.weight = nn.Parameter(torch.randn(out_features, in_features))

        # Bias parameter
        self.bias = nn.Parameter(torch.zeros(out_features))

        # NEW: Gate scores (same shape as weights)
        self.gate_scores = nn.Parameter(torch.ones_like(self.weight) * 2.0)
        # Initialize to 2.0 so sigmoid(2) ≈ 0.88

        nn.init.kaiming_normal_(self.weight)

    def forward(self, x):
        # Transform gate_scores to gates (0 to 1)
        gates = torch.sigmoid(self.gate_scores)

        # Compute pruned weights
        pruned_weights = self.weight * gates

        # Apply linear transformation
        output = F.linear(x, pruned_weights, self.bias)

        return output

In [4]:
class PrunableNet(nn.Module):
    def __init__(self):
        super(PrunableNet, self).__init__()

        # Replacing nn.Linear with PrunableLinear
        self.fc1 = PrunableLinear(3 * 32 * 32, 512)
        self.fc2 = PrunableLinear(512, 256)
        self.fc3 = PrunableLinear(256, 10)

        self.relu = nn.ReLU()

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [5]:
def compute_sparsity_loss(model):
    """
    Calculating L1 norm of all gate values.
    """
    sparsity_loss = 0.0

    for module in model.modules():
        if isinstance(module, PrunableLinear):
            # Get gates (sigmoid of gate_scores)
            gates = torch.sigmoid(module.gate_scores)

            # Add L1 norm (sum of absolute values)
            sparsity_loss += gates.sum()

    return sparsity_loss

In [6]:
classes = ('plane', 'car', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck')

transform = transforms.Compose([
    # Convert PIL Image to PyTorch Tensor (changes range from 0-255 to 0-1)
    transforms.ToTensor(),

    # Normalize with mean and std of CIFAR-10 dataset
    transforms.Normalize((0.5, 0.5, 0.5),  # Mean 
                         (0.5, 0.5, 0.5))  # Std 
])

print("\nDownloading CIFAR-10 training data...")
trainset = torchvision.datasets.CIFAR10(
    root='./data',           
    train=True,              
    download=True,           
    transform=transform      
)

100%|██████████| 170M/170M [00:01<00:00, 93.3MB/s]


In [7]:
# Download and load the test dataset
print("Downloading CIFAR-10 test data...")
testset = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,            
    download=True,
    transform=transform
)

batch_size = 64

trainloader = DataLoader(
    trainset,
    batch_size=batch_size,
    shuffle=True,            
    num_workers=2            
)

testloader = DataLoader(
    testset,
    batch_size=batch_size,
    shuffle=False,          
    num_workers=2
)

print(f"\nDataset loaded successfully!")
print(f"Training images: {len(trainset)}")
print(f"Test images: {len(testset)}")
print(f"Batch size: {batch_size}")
print(f"Number of classes: {len(classes)}")


Dataset loaded successfully!
Training images: 50000
Test images: 10000
Batch size: 64
Number of classes: 10


In [8]:
# print("\n" + "="*70)
# print("Understanding Image Structure")
# print("="*70)

# # Get a sample batch to understand the data structure
# sample_images, sample_labels = next(iter(trainloader))

# print(f"\nBatch shape: {sample_images.shape}")
# print(f"  - Batch size: {sample_images.shape[0]} images")
# print(f"  - Channels: {sample_images.shape[1]} (Red, Green, Blue)")
# print(f"  - Height: {sample_images.shape[2]} pixels")
# print(f"  - Width: {sample_images.shape[3]} pixels")
# print(f"\nEach image has {3 * 32 * 32} = 3,072 values (pixels)")
# print("We'll 'flatten' these into a 1D vector for our network")

In [9]:
def evaluate_model(model, testloader):
    """
    Evaluating the network on test data.
    
    Returns:
        accuracy: Test accuracy percentage
    """
    model.eval()  # Set to evaluation mode
    correct = 0
    total = 0
    
    with torch.no_grad():  
        for images, labels in testloader:
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    accuracy = 100 * correct / total
    return accuracy

In [10]:
def calculate_sparsity(model, threshold=1e-2):
    """
    Calculating percentage of pruned weights.

    Returns:
        sparsity_percentage: % of weights pruned
    """
    total_weights = 0
    pruned_weights = 0

    for module in model.modules():
        if isinstance(module, PrunableLinear):
            gates = torch.sigmoid(module.gate_scores)

            total_weights += gates.numel()
            pruned_weights += (gates < threshold).sum().item()

    sparsity_percentage = 100 * pruned_weights / total_weights
    return sparsity_percentage

In [11]:
# def get_lambda_schedule(epoch, max_epochs, final_lambda):
#     """Gradually increase lambda over training."""
#     # Start with 0, ramp up to final_lambda over first half of training
#     if epoch < max_epochs // 2:
#         return final_lambda * (epoch / (max_epochs // 2))
#     else:
#         return final_lambda

def get_lambda_schedule(epoch, max_epochs, final_lambda):
    # Ramp over first 2/3 of training instead of first half
    warmup_epochs = int(max_epochs * 0.67)
    if epoch < warmup_epochs:
        return final_lambda * (epoch / warmup_epochs)
    else:
        return final_lambda

In [12]:
# Training parameters
num_epochs = 30
# lambda_values = [0, 1e-6, 1e-5,1e-4,3e-05, 7e-05]  
lambda_values = [1e-5]  
results = []

for lambda_param in lambda_values:
    print("\n" + "="*70)
    print(f"Training with λ = {lambda_param}")
    print("="*70)
    
    # fresh model for each lambda
    model = PrunableNet()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()
    
    # Training loop
    model.train()
    for epoch in range(num_epochs):

        current_lambda = get_lambda_schedule(epoch, num_epochs, lambda_param)
        
        running_ce_loss = 0.0
        running_sparse_loss = 0.0
        running_total_loss = 0.0
        correct = 0
        total = 0
        
        for batch_idx, (images, labels) in enumerate(trainloader):
            # Forward pass
            outputs = model(images)
            
            # Classification loss
            ce_loss = criterion(outputs, labels)
            
            # Sparsity loss
            sparse_loss = compute_sparsity_loss(model)
            
            # Total loss
            total_loss = ce_loss + current_lambda * sparse_loss
            
            # Backward pass
            optimizer.zero_grad()
            total_loss.backward()
            optimizer.step()
            
            # Track metrics
            running_ce_loss += ce_loss.item()
            running_sparse_loss += sparse_loss.item()
            running_total_loss += total_loss.item()
            
            # Calculate accuracy
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            # Print progress every 100 batches
            if (batch_idx + 1) % 100 == 0:
                avg_ce = running_ce_loss / 100
                avg_sparse = running_sparse_loss / 100
                avg_total = running_total_loss / 100
                accuracy = 100 * correct / total
                
                print(f'Epoch [{epoch+1:2d}/{num_epochs}], '
                      f'Batch [{batch_idx+1:3d}/{len(trainloader)}], '
                      f'λ: {current_lambda:.2e}, '
                      f'CE Loss: {avg_ce:.4f}, '
                      f'Sparse: {avg_sparse:.2f}, '
                      f'Total: {avg_total:.4f}, '
                      f'Acc: {accuracy:.2f}%')
                
                running_ce_loss = 0.0
                running_sparse_loss = 0.0
                running_total_loss = 0.0
        
        # epoch summary
        epoch_accuracy = 100 * correct / total
        sparsity = calculate_sparsity(model)
        print(f'✓ Epoch {epoch+1} Complete - λ: {current_lambda:.2e}, '
              f'Train Acc: {epoch_accuracy:.2f}%, Sparsity: {sparsity:.2f}%')
    
    # Evaluating on test set
    print(f"\n📊 Evaluating model with λ = {lambda_param}...")
    test_accuracy = evaluate_model(model, testloader)
    final_sparsity = calculate_sparsity(model)
    
    print(f"Test Accuracy: {test_accuracy:.2f}%")
    print(f"Final Sparsity: {final_sparsity:.2f}%")
    
    # results
    results.append({
        'lambda': lambda_param,
        'accuracy': test_accuracy,
        'sparsity': final_sparsity,
        'model': model  # Save model for later visualization
    })

# final results table
print("\n" + "="*70)
print("FINAL RESULTS: Sparsity-Accuracy Trade-off")
print("="*70)
print(f"{'Lambda':<15} {'Test Accuracy (%)':<20} {'Sparsity Level (%)':<20}")
print("-"*70)
for r in results:
    print(f"{r['lambda']:<15.0e} {r['accuracy']:<20.2f} {r['sparsity']:<20.2f}")
print("="*70)


Training with λ = 1e-05
Epoch [ 1/30], Batch [100/782], λ: 0.00e+00, CE Loss: 1.9237, Sparse: 1502917.08, Total: 1.9237, Acc: 32.91%
Epoch [ 1/30], Batch [200/782], λ: 0.00e+00, CE Loss: 1.7616, Sparse: 1502731.85, Total: 1.7616, Acc: 35.84%
Epoch [ 1/30], Batch [300/782], λ: 0.00e+00, CE Loss: 1.7048, Sparse: 1502593.32, Total: 1.7048, Acc: 37.03%
Epoch [ 1/30], Batch [400/782], λ: 0.00e+00, CE Loss: 1.6137, Sparse: 1502465.75, Total: 1.6137, Acc: 38.58%
Epoch [ 1/30], Batch [500/782], λ: 0.00e+00, CE Loss: 1.6055, Sparse: 1502352.08, Total: 1.6055, Acc: 39.38%
Epoch [ 1/30], Batch [600/782], λ: 0.00e+00, CE Loss: 1.5486, Sparse: 1502231.96, Total: 1.5486, Acc: 40.36%
Epoch [ 1/30], Batch [700/782], λ: 0.00e+00, CE Loss: 1.5634, Sparse: 1502126.91, Total: 1.5634, Acc: 41.01%
✓ Epoch 1 Complete - λ: 0.00e+00, Train Acc: 41.27%, Sparsity: 0.00%
Epoch [ 2/30], Batch [100/782], λ: 5.00e-07, CE Loss: 1.4590, Sparse: 1501596.14, Total: 2.2098, Acc: 48.81%
Epoch [ 2/30], Batch [200/782], λ:

In [13]:
# Visualize gate distribution for each model
def plot_gate_distribution(model, lambda_param, filename):
    """Plot histogram of gate values."""
    all_gates = []
    
    for module in model.modules():
        if isinstance(module, PrunableLinear):
            gates = torch.sigmoid(module.gate_scores)
            all_gates.extend(gates.detach().cpu().numpy().flatten())
    
    plt.figure(figsize=(10, 6))
    plt.hist(all_gates, bins=50, color='steelblue', edgecolor='black', alpha=0.7)
    plt.xlabel('Gate Value', fontsize=12)
    plt.ylabel('Frequency', fontsize=12)
    plt.title(f'Gate Distribution (λ = {lambda_param:.0e})', fontsize=14)
    plt.axvline(x=0.01, color='red', linestyle='--', linewidth=2, label='Threshold (0.01)')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.savefig(filename, dpi=100, bbox_inches='tight')
    plt.close()
    print(f"✓ Saved gate distribution to '{filename}'")


print("\n📊 Creating visualizations...")
for r in results:
    filename = f"gate_dist_lambda_{r['lambda']:.0e}.png"
    plot_gate_distribution(r['model'], r['lambda'], filename)


📊 Creating visualizations...
✓ Saved gate distribution to 'gate_dist_lambda_1e-05.png'
